# Bonus Challenge 5: Deploying Agents to Agent Engine
Deploy an ADK agent to Google Agent Engine and test it remotely.

In [6]:
!pip install google-adk google-cloud-aiplatform[agent_engines,adk]

## Setup imports and environment

In [7]:
import os
import vertexai
from google.adk import Agent
from google.adk.tools import google_search
from vertexai.preview.reasoning_engines import AdkApp
from vertexai import agent_engines

PROJECT_ID = "qwiklabs-gcp-01-fab96dd167ae"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://challenge_5_bucket"  # Create a GCS bucket and put its name here

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

## Define the agent
Using a simple search agent for deployment.

In [8]:
search_agent = Agent(
    name="search_agent",
    model="gemini-2.0-flash-lite",
    description="A search agent that answers questions using Google Search.",
    instruction="""You are a helpful assistant. Use Google Search to find accurate,
up-to-date information and answer the user's question clearly and concisely.""",
    tools=[google_search],
)

## Step 1: Test the agent locally
Wrap in AdkApp and test with stream_query before deploying.

In [9]:
app = AdkApp(agent=search_agent)

for event in app.stream_query(
    user_id="test_user",
    message="What are the latest headlines in technology?",
):
    print(event)

{'model_version': 'gemini-2.0-flash-lite', 'content': {'parts': [{'text': 'Here are some of the latest headlines in technology:\n\n*   **Apple:** Apple is reportedly adding AI disclosures to music on Apple Music and has released new OS updates to prepare Macs for new displays. The new MacBook Neo is here, running the A18 Pro mobile chip.\n*   **SpaceX:** SpaceX has made "Starlink Mobile" official.\n*   **AI:** OpenAI\'s new GPT-5.4 outperforms humans in pro-level work in tests by 83%.\n*   **Amazon:** Amazon is making a new investment in healthcare AI, competing with Microsoft in the doctor\'s office.\n*   **Security:** A hacker breached YggTorrent.\n*   **Other:** TCL is teasing a Mini LED gaming monitor that can hit a 1,040Hz refresh rate.\n\n'}], 'role': 'model'}, 'grounding_metadata': {'grounding_chunks': [{'web': {'domain': 'pcmag.com', 'title': 'pcmag.com', 'uri': 'https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHc4lcjcDNRqD0UCtBANcfNLhNBCjUL8qi-MbMRm17EkdNFv

## Step 2: Deploy to Agent Engine
This takes approximately 10 minutes.

In [10]:
remote_agent = agent_engines.create(
    app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)

print(f"Deployed! Resource name: {remote_agent.resource_name}")

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'google-cloud-aiplatform': '1.135.0', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket challenge_5_bucket
INFO:vertexai.agent_engines:Wrote to gs://challenge_5_bucket/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://challenge_5_bucket/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://challenge_5_bucket/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/528312607381/locations/us-central1/reasoningEngines/4030

Deployed! Resource name: projects/528312607381/locations/us-central1/reasoningEngines/4030834916194254848


## Step 3: Test the deployed agent
Query the remote agent to prove it works.

In [11]:
for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="What is the current weather news in the United States?",
):
    print(event)

{'model_version': 'gemini-2.0-flash-lite', 'content': {'parts': [{'text': "The current weather news in the United States includes:\n\n*   **Severe weather threats:** Millions of people in Texas, Oklahoma, the Plains, and Midwest are facing a multi-day severe weather threat, with the greatest risk of large hail, damaging wind gusts, and tornadoes on Friday. The Central Plains are expected to see severe storms that could bring damaging hail, wind, and possible tornadoes next week.\n*   **Earthquake:** A rare magnitude 4.9 earthquake struck rural Louisiana, marking the strongest in the state's history.\n*   **Storms and Flooding:** Storms are easing drought conditions in Mississippi and the Ohio Valley, but flash flood threats are increasing, leading to water rescues.\n*   **Other weather events:** There's a forecast for snow and ice in parts of New York and New England.\n*   **Long-term trends**: Climate change is influencing US weather patterns, leading to more intense hurricanes, prolo

## Step 4: Test with another query

In [12]:
for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="What are the top 3 AI developments this week?",
):
    print(event)

{'model_version': 'gemini-2.0-flash-lite', 'content': {'parts': [{'text': 'Here are some of the top AI developments this week, based on the search results:\n\n1.  **AI in War Simulations:** A study showed that AI models, when given nuclear launch codes in simulated Cold War-style crises, deployed nukes in 95% of scenarios. The AI models developed their own distinct personalities and strategies, with no de-escalation options being utilized.\n2.  **Job Market Impact:** There are reports of companies laying off employees due to the implementation of AI tools, which is raising concerns about the potential impact of AI on jobs. Jack Dorsey, for example, stated that smaller teams can now do more with AI. However, there are also arguments that the impact of AI on jobs is not as drastic as some predict.\n3.  **Pentagon and AI:** There was a dispute between Anthropic and the Pentagon regarding the use of AI for mass surveillance and autonomous weapons. OpenAI then cut its own deal with the Pent

## Cleanup (optional)
Delete the deployed agent when done to avoid charges.

In [13]:
# Uncomment to delete:
# remote_agent.delete()